# Folium

## Imports

In [ ]:
import json

import folium
import geopandas

## Initial Settings and Displaying a Map

Get the GPS coordinates of Nantes (on Wikipedia, for instance), then display a Folium map centered on these coordinates with a zoom level that shows the city well.

In [ ]:
import pathlib
# Your code here

### Solution

In [ ]:
NANTES_COORDS = 47.218371, -1.553621

folium.Map(location=NANTES_COORDS, zoom_start=14)

## Adding Markers

Now that we have a properly initialized map, we'll display on it usage information of Nantes' bike-sharing service: bicloo.

Let's start by fetching the data with the cells below.

In [ ]:
!wget https://github.com/shuuchuu/datasets/raw/refs/heads/main/nantes-open-data/infos-velos.json

### Loading the Data

This data is now available in the `infos-velos.json` file. Load this file with the standard library's `json` module and display the loaded data.

In [ ]:
# Your code here

#### Solution

In [ ]:
with pathlib.Path("infos-velos.json").open(encoding="utf8") as fh:
  content = json.load(fh)
print(content)

### Creating the Markers

You can now iterate over the data (in particular over the content under the `records` key of each data element) to display a marker at each bicloo station position.

In [ ]:
# Your code here

#### Solution

In [ ]:
m = folium.Map(location=NANTES_COORDS, zoom_start=14)

for record in content["records"]:
  folium.Marker(location=record["fields"]["position"]).add_to(m)

m

### Customizing the Markers

Add some information to these markers:

- The color must depend on the number of bikes:
    - 0 → red
    - < 4 → yellow
    - otherwise green
- A station's name is displayed when clicking a marker.

In [ ]:
# Your code here

#### Solution

In [ ]:
m = folium.Map(location=NANTES_COORDS, zoom_start=14)

for record in content["records"]:
  if record["fields"]["status"] != "OPEN":
    color = "black"
  else:
    match record["fields"]["available_bikes"]:
      case x if x > 3:
        color = "green"
      case x if x > 0:
        color = "yellow"
      case _:
        color = "red"
  folium.Marker(
    location=record["fields"]["position"],
    popup=record["fields"]["name"],
    icon=folium.Icon(color=color),
  ).add_to(m)

m

## Adding a GeoJSON Layer

We'll now create a more complex map, with a GeoJSON layer.

To do so, we'll fetch two GeoJSON files: one contains all the polling stations of Nantes, the other the voting areas.

In [ ]:
!wget https://github.com/shuuchuu/datasets/raw/refs/heads/main/nantes-open-data/zones.geojson
!wget https://github.com/shuuchuu/datasets/raw/refs/heads/main/nantes-open-data/bureaux.geojson

### Loading the Data

Start by loading the fetched data with the [`geopandas.read_file`](https://geopandas.org/en/stable/docs/reference/api/geopandas.read_file.html) function.

Then join the two tables for later use. You can use the `lien_lieu_vote` column of the areas table and the `idlieu_vote` column of the polling stations table. The join function is [`pandas.DataFrame.join`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.join.html).

In [ ]:
# Your code here

#### Solution

In [ ]:
bureaux_gdf = geopandas.read_file("bureaux.geojson")
zones_gdf = geopandas.read_file("zones.geojson")
fat_gdf = zones_gdf.join(
  bureaux_gdf.set_index("idlieu_vote"), on="lien_lieu_vote", rsuffix="_bureau"
)

### Displaying the Voting Areas

Use [`folium.GeoJson`](https://python-visualization.github.io/folium/modules.html#folium.features.GeoJson) to display the voting areas.

For an extra challenge, display the area's name and the number of the polling station linked to the area on mouse hover.

In [ ]:
# Your code here

#### Solution

In [ ]:
m = folium.Map(location=NANTES_COORDS, zoom_start=13)

folium.GeoJson(
  fat_gdf.drop(columns=["geometry_bureau"]).set_geometry("geometry"),
  tooltip=folium.GeoJsonTooltip(
    fields=["nom", "numero_bureau"], aliases=["Area", "Polling station no."]
  ),
  name="Voting areas",
).add_to(m)

folium.LayerControl().add_to(m)

m

### Adding Markers for the Polling Stations

Start from the code of the previous map and add markers for each polling station. For an extra challenge, display the station's name and address when clicking the marker.

In [ ]:
# Your code here

#### Solution

In [ ]:
m = folium.Map(location=NANTES_COORDS, zoom_start=13)

folium.GeoJson(
  fat_gdf.drop(columns=["geometry_bureau"]).set_geometry("geometry"),
  tooltip=folium.GeoJsonTooltip(
    fields=["nom", "numero_bureau"], aliases=["Area", "Polling station no."]
  ),
  name="Voting areas",
).add_to(m)

folium.LayerControl().add_to(m)

for _, row in bureaux_gdf.iterrows():
  folium.Marker(
    location=(row.geometry.y, row.geometry.x),
    popup=f"{row.nom}<br />{row.adresse}",
    icon=folium.Icon(icon="info-sign", color="green"),
  ).add_to(m)

m